#  Player Duration Analysis (Strict Phase 2) (Strict Phase 2)

This notebook analyzes player duration using **ONLY** the following two files:
1. `data/telemetry_phase_2.telemetries.csv`
2. `data/telemetry_phase_2.users.csv`

###  Analysis Logic
1.  **Load Data**: Explicitly load the two specified CSV files.
2.  **Preprocessing**: Convert timestamps and map User IDs to Names.
3.  **Duration Calculation**: 
    - Sort by User and Timestamp.
    - Identify sessions (Gap > 120 seconds = break).
    - Sum session durations (Session End - Session Start + 30s).
4.  **Output**: Display a sorted table of players by total duration.

In [ ]:
import pandas as pd
import datetime
import os

# Settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)

##  1. Load Data
Loading strictly `telemetry_phase_2.telemetries.csv` and `telemetry_phase_2.users.csv`.

In [ ]:
# Define explicit paths
telemetry_file = os.path.join('data', 'telemetry_phase_2.telemetries.csv')
users_file = os.path.join('data', 'telemetry_phase_2.users.csv')

# Load Telemetry
if os.path.exists(telemetry_file):
    df_telemetry = pd.read_csv(telemetry_file)
    print(f"Loaded Telemetry: {len(df_telemetry)} rows")
else:
    print(f"Error: {telemetry_file} not found.")
    df_telemetry = pd.DataFrame()

# Load Users
if os.path.exists(users_file):
    df_users = pd.read_csv(users_file)
    print(f"Loaded Users: {len(df_users)} rows")
else:
    print(f"Error: {users_file} not found.")
    df_users = pd.DataFrame()

##  2. Preprocessing & Merging
Mapping `userId` to User Names.

In [ ]:
if not df_telemetry.empty:
    # Convert timestamp
    df_telemetry['timestamp'] = pd.to_datetime(df_telemetry['timestamp'], errors='coerce')
    df_telemetry = df_telemetry.dropna(subset=['timestamp'])
    
    # Prepare Users for Merge
    if not df_users.empty and '_id' in df_users.columns:
        # Create a name column
        df_users['PlayerName'] = df_users['firstName'].fillna('') + ' ' + df_users['lastName'].fillna('')
        df_users['PlayerName'] = df_users['PlayerName'].str.strip()
        
        # Merge
        df_merged = df_telemetry.merge(df_users[['_id', 'PlayerName']], left_on='userId', right_on='_id', how='left')
    else:
        df_merged = df_telemetry.copy()
        df_merged['PlayerName'] = 'Unknown'
        
    # Sort for calculation
    df_merged = df_merged.sort_values(by=['userId', 'timestamp'])
else:
    df_merged = pd.DataFrame()

##  3. Calculate Duration

In [ ]:
player_stats = []

if not df_merged.empty:
    for user_id, group in df_merged.groupby('userId'):
        group = group.sort_values('timestamp')
        
        # Get Player Name (take the first non-null one found for this ID)
        player_name = group['PlayerName'].iloc[0] if 'PlayerName' in group.columns else str(user_id)
        if pd.isna(player_name) or player_name == '':
            player_name = str(user_id)
            
        # Calculate time diffs
        group['time_diff'] = group['timestamp'].diff().dt.total_seconds()
        
        # Convert None/NaN to True for new session (first record)
        group['new_session'] = (group['time_diff'] > 120) | (group['time_diff'].isna())
        
        # Session Grouping
        group['session_group_id'] = group['new_session'].cumsum()
        
        total_duration = 0
        session_count = 0
        
        for _, session_data in group.groupby('session_group_id'):
            session_count += 1
            if len(session_data) <= 1:
                total_duration += 30
            else:
                start = session_data['timestamp'].min()
                end = session_data['timestamp'].max()
                total_duration += (end - start).total_seconds() + 30
        
        player_stats.append({
            'Player Name': player_name,
            'User ID': user_id,
            'Total Duration (s)': total_duration,
            'Total Duration (m)': round(total_duration / 60, 2),
            'Formatted Duration': str(datetime.timedelta(seconds=int(total_duration))),
            'Sessions': session_count
        })
        
    # Create DataFrame
    results_df = pd.DataFrame(player_stats)
    
    # Sort by Duration
    if not results_df.empty:
         results_df = results_df.sort_values(by='Total Duration (s)', ascending=False, ignore_index=True)
         results_df.index += 1 # 1-based rank
         results_df.index.name = 'Rank'
else:
    results_df = pd.DataFrame()

##  4. Final Sorted Table

In [ ]:
if not results_df.empty:
    print("Top Players by Duration:")
    display(results_df[['Player Name', 'Formatted Duration', 'Sessions', 'Total Duration (m)']])
else:
    print("No data available.")

## 5. Duration Restriction Analysis
1. **Exclude** players with Total Duration < `MIN_CAP_MINUTES`.
2. **Cap** players with Total Duration > `MAX_CAP_MINUTES` (only count data points within the first `MAX_CAP` minutes).

In [ ]:
# Configure duration caps (in minutes)
MIN_CAP_MINUTES = 20
MAX_CAP_MINUTES = 45

valid_data_points_count = 0
valid_players_count = 0
kept_players = []

print(f"Processing with MIN_CAP={MIN_CAP_MINUTES}m, MAX_CAP={MAX_CAP_MINUTES}m...")

if not df_merged.empty:
    # Iterate over each user
    for user_id, group in df_merged.groupby('userId'):
        group = group.sort_values('timestamp')
        
        # 1. Calculate stats required to check Total Duration first
        # (Re-using logic from cell 4 for consistency)
        group['time_diff'] = group['timestamp'].diff().dt.total_seconds()
        group['new_session'] = (group['time_diff'] > 120) | (group['time_diff'].isna())
        group['session_group_id'] = group['new_session'].cumsum()
        
        total_duration_seconds = 0
        
        # Calculate total duration
        for _, session_data in group.groupby('session_group_id'):
            if len(session_data) <= 1:
                total_duration_seconds += 30
            else:
                start = session_data['timestamp'].min()
                end = session_data['timestamp'].max()
                total_duration_seconds += (end - start).total_seconds() + 30
        
        total_duration_minutes = total_duration_seconds / 60.0
        
        # 2. Check MIN Cap
        if total_duration_minutes < MIN_CAP_MINUTES:
            continue # Drop user entirely
            
        valid_players_count += 1
        
        # Get Player Name (optional, for display)
        player_name = group['PlayerName'].iloc[0] if 'PlayerName' in group.columns else str(user_id)
        kept_players.append({'Player Name': player_name, 'Total Duration (m)': round(total_duration_minutes, 2)})
        
        # 3. Apply MAX Cap to count data points
        # We count rows until the cumulative duration hits MAX_CAP_MINUTES
        remaining_seconds = MAX_CAP_MINUTES * 60
        
        for _, session_data in group.groupby('session_group_id'):
            if remaining_seconds <= 0:
                break
                
            # Calculate this session's duration
            if len(session_data) <= 1:
                session_duration = 30
            else:
                start = session_data['timestamp'].min()
                end = session_data['timestamp'].max()
                session_duration = (end - start).total_seconds() + 30
            
            if session_duration <= remaining_seconds:
                # Take entire session
                valid_data_points_count += len(session_data)
                remaining_seconds -= session_duration
            else:
                # Take partial session
                # Logic: We have `remaining_seconds` budget.
                # We take rows that fall within `start + remaining_seconds`.
                start = session_data['timestamp'].min()
                cutoff_time = start + pd.Timedelta(seconds=remaining_seconds)
                
                # Count rows <= cutoff_time
                valid_rows = session_data[session_data['timestamp'] <= cutoff_time]
                valid_data_points_count += len(valid_rows)
                remaining_seconds = 0 # Budget exhausted

    print(f"\nResults:")
    print(f"Number of Players kept (>= {MIN_CAP_MINUTES}m): {valid_players_count}")
    print(f"Total Data Points (capped at {MAX_CAP_MINUTES}m): {valid_data_points_count}")
    
    if kept_players:
        df_kept = pd.DataFrame(kept_players)
        df_kept = df_kept.sort_values(by='Total Duration (m)', ascending=False)
        print("\nSample of Kept Players:")
        display(df_kept.head(10))
else:
    print("No data to process.")